# Valeurs manquantes

Les valeurs manquantes dans un ensemble de données, souvent représentées par `NaN`, sont des entrées absentes ou non disponibles. Elles peuvent survenir pour diverses raisons, comme des erreurs de collecte de données ou des omissions. Gérer ces valeurs est crucial pour éviter les biais dans l'analyse et garantir des résultats fiables, en utilisant des méthodes comme l'interpolation, la suppression ou le remplissage des données manquantes.

In [ ]:
import pandas as pd
import numpy as np

Les datasets que vous rencontrerez en pratique contiendront (très) souvent des valeurs manquantes (ex : capteur en panne, erreur humaine, calcul invalide dans la préparation des données...).

Le dataset de cette section contient des informations à propos des accidents de voiture à New York.

In [ ]:
# Comme d'habitude, inspectons le dataframe avec .head()
collisions = pd.read_csv('/content/collisions.csv', sep=';')
collisions.head()

### Compter les valeurs manquantes

Nous comptons le nombre de valeurs manquantes avec `.isna().sum()`.

La méthode **`.isna()`** en pandas **identifie les valeurs manquantes** dans un DataFrame ou une Series, en retournant un objet booléen de la même forme, avec True pour les valeurs NaN ou nulles et False pour les autres valeurs. Elle est souvent utilisée pour détecter, filtrer ou traiter les données manquantes.

In [ ]:
collisions.isna()

La méthode **`.isna().sum()`** en pandas **calcule le nombre total de valeurs manquantes dans chaque colonne** d'un DataFrame ou dans une Series. D'abord, `.isna()` détecte les valeurs manquantes et retourne un DataFrame ou une Series de booléens (`True` pour les valeurs NaN). Ensuite, `.sum()` additionne ces booléens (`True` étant compté comme 1 et False comme 0), fournissant le nombre de valeurs manquantes par colonne ou pour toute la Series.

In [ ]:
collisions.isna().sum()

La méthode **`.isna().mean()`** en pandas **calcule la proportion de valeurs manquantes dans chaque colonne** d'un DataFrame ou dans une Series. D'abord, `.isna()` identifie les valeurs manquantes et retourne un DataFrame ou une Series de booléens (True pour les valeurs NaN). Ensuite, `.mean()` calcule la moyenne de ces booléens (True étant équivalent à 1 et False à 0), ce qui donne la proportion des valeurs manquantes pour chaque colonne ou pour la Series.

In [ ]:
collisions.isna().mean()

Pour aller plus loin : Une 2ème méthode pour repérer les valeurs manquantes consiste à utiliser une propriété des NaNs.

PS: Cette méthode n'est jamais utilisée mais vous permet de mieux comprendre le fait qu'un `Nan` n'est pas égal à lui même.

In [ ]:
np.nan

In [ ]:
np.nan > 0, np.nan < 0, np.nan == 0

In [ ]:
np.nan == np.nan

`np.nan` est la seule valeur qui n'est pas égale à elle même ! Cette propriété est l'une des méthodes pour répérer les NaNs dans un dataset.

In [ ]:
collisions['BOROUGH'] == collisions['BOROUGH']

In [ ]:
(collisions['BOROUGH'] == collisions['BOROUGH']).sum()  # Somme de tous les True trouvés. Si Nan alors False et donc pas compté dans la somme

Nous sommons tous les `True` trouvés dans notre colonne "BOROUGH". Si la valeur correspond à un `Nan` alors il y aura `False` car comme vu précédemment, un `Nan` n'est pas égal à lui même. La somme nous comptera donc seulement les valeurs `True`, qui ne sont pas `Nan`.

In [ ]:
len(collisions) - (collisions['BOROUGH'] == collisions['BOROUGH']).sum()

Au vu de l'explication précédente, nous pouvons ainsi soustraire cette somme à la longueur de notre DataFrame afin d'avoir ainsi le nombre de valeurs manquantes.

_Visualisation_

Le package **`missingno`** permet d'afficher les valeurs manquantes de manière plus visuelle !

In [ ]:
!pip install missingno

In [ ]:
import missingno as msno

In [ ]:
msno.matrix(collisions.sample(250))

### Gérer les valeurs manquantes

Quelles sont nos options ?
* Supprimer
* Remplacer

Pas de réponse exacte, le choix dépend des données et du use case !

Il est donc plus qu'important de **connaître son jeu de données** ainsi que la/les **problématique(s)** que nous avons.

Exemple : les valeurs manquantes de la colonne `BOROUGH`.

In [ ]:
missing_boroughs = collisions["BOROUGH"].isna().mean()
print(f"{missing_boroughs: .2%} of boroughs are missing.")

#### Il est possible de supprimer ces valeurs manquantes.

Supprimer les valeurs manquantes est essentiel pour **éviter les biais** ou **erreurs** dans les analyses et modèles, et pour **simplifier le traitement des données**. Retirer les lignes ou colonnes contenant des valeurs NaN, assure que seules les données complètes sont utilisées dans les calculs et visualisations.

La méthode **`.dropna()`** est utilisée pour **supprimer les lignes ou les colonnes contenant des valeurs manquantes (NaN)** dans un DataFrame ou une série de données en pandas. On peut spécifier si l'on veut *supprimer les lignes* ou *les colonnes*, et on peut aussi choisir de ne supprimer que les lignes/colonnes où toutes les valeurs sont manquantes, ou dès qu'il y en a au moins une de manquante.

In [ ]:
# Supprimons les lignes contenant des nans dans la colonne "BOROUGH" : méthode .dropna()

collisions2 = collisions.dropna(subset=['BOROUGH'])

Nous pouvons constater que nous n'avons plus de NaN dans la colonne "BOROUGH".

In [ ]:
collisions2.isna().mean()

#### Il est possible de remplacer ces valeurs manquantes.

Remplacer les valeurs manquantes est crucial pour assurer la **précision** des analyses et des modèles, **éviter les erreurs** de calculs, et obtenir des **résultats fiables**.




In [ ]:
# Essayons de remplacer les nans de façon plus pertinente.
# Quelle est la distribution des quartiers dans notre dataframe ?

collisions["BOROUGH"].value_counts()

La méthode `.fillna()` est utilisée pour **remplacer les valeurs manquantes (NaN)** dans un DataFrame ou une série de données par une **valeur spécifique**. On peut remplir ces valeurs avec un nombre, du texte (moyenne, max, min, valeur la plus commune, texte spécifique), une méthode comme ffill (remplissage avec la valeur précédente) ou bfill (remplissage avec la valeur suivante), ou encore par d'autres valeurs dérivées des données.

In [ ]:
# Nous remplaçons nos NaNs par "UNKNOWN".

collisions["BOROUGH"].fillna('UNKNOWN', inplace= True)  # inplace= True permet de l'ancrer dans le DataFrame ou la série.

In [ ]:
# Equivalent à :
collisions["BOROUGH"] = collisions["BOROUGH"].fillna('UNKNOWN')

In [ ]:
collisions["BOROUGH"].isna().mean()

Il y a également le cas de données cycliques ou qui suivent un certain pattern.

In [ ]:
# Créons des données fictives.
temp_uganda = pd.Series([19.49166667,  19.50833333,  19.44166667,  20.41666667,  22.34166667,  23.25833333,  24.775,  25.10833333,  25.83333333,  26.05833333,  25.64166667,  24.9,  23.35,  21.96666667,  20.73333333,  19.76666667,  19.375,  19.09166667,  18.2,  18.71666667,  18.90833333,  19.09166667,  19.14166667,  19.16666667,  19.23333333,  19.73333333,  19.81666667,  19.94166667,  20.925,  23.76666667,  25.48333333,  26.7,  26.875,  25.425,  21.40833333,  23.225,  24.35,  24.18333333,  21.325,  19.51666667,  19.61666667,  18.98333333,  18.94166667,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  23.99090909,  25.30833333,  26.35,  26.91666667,  27.03636364,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  np.nan,  20.56363636,  19.93333333,  19.79166667,  19.525,  19.575,  19.475,  19.475,  19.90833333,  21.83333333,  23.375,  24.71666667,  25.225,  26.125,  26.9,  26.975,  26.475,  25.38333333,  24.21666667,  23.025,  22.625,  22.25833333,  21.9,  21.375,  21.4,  20.89166667,  20.90833333,  21.275,  21.44166667,  21.175,  20.61666667])

In [ ]:
temp_uganda.plot() # Faisons une visualisation rapide.

On pourrait remplacer les NaNs avec la moyenne des valeurs, mais ça ne serait pas très réaliste :

In [ ]:
# Nous remplaçons nos données manquantes par la moyenne et traçons le graphique.
temp_uganda.fillna(temp_uganda.mean()).plot()

Ici, une méthode qui pourrait être pertinente serait la méthode `.interpolate()`.

La méthode **`.interpolate()`** est utilisée pour **remplacer les valeurs manquantes** (NaN) dans un DataFrame ou une série de données en interpolant les données environnantes. Elle **calcule et remplit les valeurs manquantes en fonction des données adjacentes**, selon une méthode spécifiée, comme linéaire, polynomiale, ou basée sur le temps.

In [ ]:
# Nous appliquons la méthode interpolate à notre jeu de données et traçons le graphique.
temp_uganda.interpolate().plot()

La méthode `.interpolate()`comprend un paramètre qui se nomme `method`. Pour ce paramètre, différents arguments existent :
* `method='linear'` : Effectue une interpolation linéaire entre les points adjacents. C'est la méthode par défaut.
* `method='time'` : Interpole en utilisant l'index temporel, idéal pour les séries temporelles.
* `method='polynomial'` : Utilise un polynôme de degré spécifié pour l'interpolation, offrant une courbe plus complexe (ex. `method= 'polynomial', order= 2`). D'ailleurs, `method= 'quadratic'` est équivalente à `method= 'polynomial', order= 2`
* `method='nearest'` : Remplace les valeurs manquantes par la valeur la plus proche.


Un des aspects les plus important avant de choisir une méthode est de bien comprendre son fonctionnement ainsi que les données que nous avons à disposition.


In [ ]:
temp_uganda.interpolate(method='quadratic').plot() # Meilleure solution !

**Conclusion** : il est important de connaître ses données avant de choisir la méthode de gestion des NaNs à employer.

**$\rightarrow$ Variables quantitatives**

* Supprimer les lignes contenant des NaNs.
* Remplacer par la moyenne de la colonne.
* Remplacer par la valeur la plus "réaliste" (médiane ? max ? à définir au cas par cas).


**$\rightarrow$ Variables qualitatives (catégorielles)**

* Supprimer les lignes contenant des NaNs.
* Remplacer par le mode (valeur la plus fréquente).
* Remplacer par un autre nom de catégorie ("autre", "inconnu"), celui que vous souhaitez.


**$\rightarrow$ Les données cycliques**

* Remplacer par la même valeur à la période précédente.
* Utiliser une interpolation.